In [ ]:
"""
MNIST 온디바이스 AI 경량화 기법 종합 비교
- 양자화 (Quantization): FP32 → FP16 → INT8
- 프루닝 (Pruning): 50% 가중치 제거
- 지식 증류 (Knowledge Distillation): Teacher → Student
"""

import tensorflow as tf
from tensorflow import keras
import tensorflow_model_optimization as tfmot
import numpy as np
import time
import os
from pathlib import Path

print(f"TensorFlow 버전: {tf.__version__}")
print(f"NumPy 버전: {np.__version__}")

# 디렉토리 생성
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

# ============================================================================
# 1. 데이터 로드 및 전처리
# ============================================================================
print("\n" + "="*80)
print("1. MNIST 데이터 로드 및 전처리")
print("="*80)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 정규화 및 reshape
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# 검증 데이터 분리
x_val = x_train[-5000:]
y_val = y_train[-5000:]
x_train = x_train[:-5000]
y_train = y_train[:-5000]

print(f"학습 데이터: {x_train.shape}")
print(f"검증 데이터: {x_val.shape}")
print(f"테스트 데이터: {x_test.shape}")

# ============================================================================
# 2. Teacher 모델 (큰 모델) 생성 및 학습
# ============================================================================
print("\n" + "="*80)
print("2. Teacher 모델 (큰 모델) 생성 및 학습")
print("="*80)

def create_teacher_model():
    """Teacher 모델 - 896K 파라미터"""
    model = keras.Sequential([
        keras.layers.Conv2D(64, 3, activation='relu', input_shape=(28, 28, 1)),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(128, 3, activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(10, activation='softmax')
    ], name='Teacher')
    return model

teacher_model = create_teacher_model()
teacher_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n[Teacher 모델 구조]")
teacher_model.summary()

print("\nTeacher 모델 학습 시작...")
teacher_model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=1
)

teacher_loss, teacher_accuracy = teacher_model.evaluate(x_test, y_test, verbose=0)
print(f"\n✅ Teacher 정확도: {teacher_accuracy*100:.2f}%")

teacher_model.save('models/teacher_model.h5')

# ============================================================================
# 3. 기본 모델 (중간 크기) 생성 및 학습
# ============================================================================
print("\n" + "="*80)
print("3. 기본 모델 (중간 크기) 생성 및 학습")
print("="*80)

def create_base_model():
    """기본 모델 - 225K 파라미터"""
    model = keras.Sequential([
        keras.layers.Conv2D(32, 3, activation='relu', input_shape=(28, 28, 1)),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation='softmax')
    ], name='Base')
    return model

base_model = create_base_model()
base_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n[기본 모델 구조]")
base_model.summary()

print("\n기본 모델 학습 시작...")
base_model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=1
)

base_loss, base_accuracy = base_model.evaluate(x_test, y_test, verbose=0)
print(f"\n✅ 기본 모델 정확도: {base_accuracy*100:.2f}%")

base_model.save('models/base_model.h5')

# ============================================================================
# 4. 프루닝 (Pruning) 적용
# ============================================================================
print("\n" + "="*80)
print("4. 프루닝 (Pruning) 적용")
print("="*80)

print("\n[1단계] 프루닝용 모델 사전 학습...")
pruning_model = create_base_model()
pruning_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

pruning_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=0
)

pre_prune_loss, pre_prune_accuracy = pruning_model.evaluate(x_test, y_test, verbose=0)
print(f"✅ 프루닝 전 정확도: {pre_prune_accuracy*100:.2f}%")

print("\n[2단계] 프루닝 적용 (50% 가중치 제거)...")
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=2000
    )
}

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(pruning_model, **pruning_params)
pruned_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n[3단계] 프루닝 모델 파인튜닝...")
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=0
)

pruned_loss, pruned_accuracy = pruned_model.evaluate(x_test, y_test, verbose=0)
print(f"✅ 프루닝 후 정확도: {pruned_accuracy*100:.2f}%")

print("\n[4단계] 프루닝 확정...")
final_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
final_pruned_model.save('models/pruned_model.h5')

pruned_h5_size = os.path.getsize('models/pruned_model.h5') / 1024
print(f"Pruned H5 크기: {pruned_h5_size:.2f}KB")

# 프루닝 모델 TFLite 변환
converter_pruned = tf.lite.TFLiteConverter.from_keras_model(final_pruned_model)
converter_pruned.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_pruned = converter_pruned.convert()
with open('models/pruned_model.tflite', 'wb') as f:
    f.write(tflite_pruned)
print(f"✅ 프루닝 모델 TFLite: {len(tflite_pruned)/1024:.2f}KB")

tflite_size = len(tflite_pruned) / 1024
print(f"Pruned TFLite 크기: {tflite_size:.2f}KB")z

# ============================================================================
# 5. 지식 증류 (Knowledge Distillation)
# ============================================================================
print("\n" + "="*80)
print("5. 지식 증류 (Knowledge Distillation) 적용")
print("="*80)

class DistillationModel(keras.Model):
    """지식 증류를 위한 커스텀 모델"""
    def __init__(self, teacher, student):
        super().__init__()
        self.teacher = teacher
        self.student = student
        
    def compile(self, optimizer, metrics, student_loss_fn, distillation_loss_fn, alpha=0.1, temperature=3):
        super().compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn
        self.alpha = alpha
        self.temperature = temperature
        
    def train_step(self, data):
        x, y = data
        teacher_predictions = self.teacher(x, training=False)
        
        with tf.GradientTape() as tape:
            student_predictions = self.student(x, training=True)
            student_loss = self.student_loss_fn(y, student_predictions)
            distillation_loss = self.distillation_loss_fn(
                tf.nn.softmax(teacher_predictions / self.temperature, axis=1),
                tf.nn.softmax(student_predictions / self.temperature, axis=1)
            ) * (self.temperature ** 2)
            total_loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss
            
        trainable_vars = self.student.trainable_variables
        gradients = tape.gradient(total_loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        self.compiled_metrics.update_state(y, student_predictions)
        
        results = {m.name: m.result() for m in self.metrics}
        results.update({
            "student_loss": student_loss,
            "distillation_loss": distillation_loss,
            "total_loss": total_loss
        })
        return results
    
    def test_step(self, data):
        x, y = data
        y_prediction = self.student(x, training=False)
        self.compiled_metrics.update_state(y, y_prediction)
        return {m.name: m.result() for m in self.metrics}

def create_student_model():
    """Student 모델 - 57K 파라미터 (Teacher의 1/16)"""
    model = keras.Sequential([
        keras.layers.Conv2D(16, 3, activation='relu', input_shape=(28, 28, 1)),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(32, 3, activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ], name='Student')
    return model

print("\n[1단계] Student 모델 생성...")
student_model = create_student_model()
print("\n[Student 모델 구조]")
student_model.summary()

print("\n[2단계] Student 단독 학습 (비교용)...")
student_scratch = create_student_model()
student_scratch.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

student_scratch.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=0
)

scratch_loss, scratch_accuracy = student_scratch.evaluate(x_test, y_test, verbose=0)
print(f"✅ Student 단독 학습 정확도: {scratch_accuracy*100:.2f}%")

print("\n[3단계] 지식 증류로 Student 학습...")
distillation_model = DistillationModel(teacher=teacher_model, student=student_model)
distillation_model.compile(
    optimizer=keras.optimizers.Adam(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
    student_loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    distillation_loss_fn=keras.losses.KLDivergence(),
    alpha=0.1,
    temperature=3
)

distillation_model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=0
)

# Student 모델로 직접 평가
student_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
distilled_loss, distilled_accuracy = student_model.evaluate(x_test, y_test, verbose=0)
print(f"✅ 지식 증류 후 정확도: {distilled_accuracy*100:.2f}%")
print(f"💡 증류 효과: {(distilled_accuracy - scratch_accuracy)*100:+.2f}%p")

student_model.save('models/student_model.h5')

# ============================================================================
# 6. 양자화 (Quantization) 변환
# ============================================================================
print("\n" + "="*80)
print("6. 양자화 (Quantization) 변환")
print("="*80)

print("\n[기본 모델 양자화]")

# FP32 TFLite
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(base_model)
tflite_fp32 = converter_fp32.convert()
with open('models/base_fp32.tflite', 'wb') as f:
    f.write(tflite_fp32)
print(f"  FP32: {len(tflite_fp32)/1024:.2f}KB")

# FP16 TFLite
converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(base_model)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter_fp16.convert()
with open('models/base_fp16.tflite', 'wb') as f:
    f.write(tflite_fp16)
print(f"  FP16: {len(tflite_fp16)/1024:.2f}KB ({((len(tflite_fp32)-len(tflite_fp16))/len(tflite_fp32)*100):.1f}% 감소)")

# INT8 TFLite
def representative_dataset():
    for i in range(100):
        yield [x_train[i:i+1]]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(base_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8
tflite_int8 = converter_int8.convert()
with open('models/base_int8.tflite', 'wb') as f:
    f.write(tflite_int8)
print(f"  INT8: {len(tflite_int8)/1024:.2f}KB ({((len(tflite_fp32)-len(tflite_int8))/len(tflite_fp32)*100):.1f}% 감소)")

print("\n[Student 모델 양자화]")
converter_student = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter_student.optimizations = [tf.lite.Optimize.DEFAULT]
converter_student.target_spec.supported_types = [tf.float16]
tflite_student = converter_student.convert()
with open('models/student_fp16.tflite', 'wb') as f:
    f.write(tflite_student)
print(f"  Student FP16: {len(tflite_student)/1024:.2f}KB")

# ============================================================================
# 7. 성능 측정 함수
# ============================================================================
print("\n" + "="*80)
print("7. 성능 측정")
print("="*80)

def count_parameters(model):
    """모델 파라미터 수 계산"""
    return sum([np.prod(var.shape) for var in model.trainable_variables])

def evaluate_tf_model_speed(model, iterations=100):
    """TensorFlow 모델 속도 측정"""
    # 워밍업
    for _ in range(10):
        model.predict(x_test[:1], verbose=0)
    
    # 측정
    start = time.time()
    for _ in range(iterations):
        model.predict(x_test[:1], verbose=0)
    end = time.time()
    
    return (end - start) / iterations * 1000

def evaluate_tflite_speed(tflite_path, iterations=1000):
    """TFLite 모델 속도 측정 (마이크로초)"""
    if not os.path.exists(tflite_path):
        return None
    
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    input_dtype = input_details[0]['dtype']
    
    # 다양한 입력 준비
    test_images = []
    for i in range(min(100, len(x_test))):
        test_image = x_test[i:i+1]
        if input_dtype == np.uint8:
            input_scale, input_zero_point = input_details[0]['quantization']
            test_image = test_image / input_scale + input_zero_point
            test_image = test_image.astype(np.uint8)
        test_images.append(test_image)
    
    # 워밍업
    for i in range(20):
        interpreter.set_tensor(input_details[0]['index'], test_images[i % len(test_images)])
        interpreter.invoke()
    
    # 측정
    start = time.time()
    for i in range(iterations):
        interpreter.set_tensor(input_details[0]['index'], test_images[i % len(test_images)])
        interpreter.invoke()
    end = time.time()
    
    return (end - start) / iterations * 1000000  # μs

print("\n[TensorFlow 모델 속도 (밀리초)]")
teacher_speed = evaluate_tf_model_speed(teacher_model)
print(f"  Teacher:  {teacher_speed:.2f}ms")

base_speed = evaluate_tf_model_speed(base_model)
print(f"  Base:     {base_speed:.2f}ms")

pruned_speed = evaluate_tf_model_speed(final_pruned_model)
print(f"  Pruned:   {pruned_speed:.2f}ms")

student_speed = evaluate_tf_model_speed(student_model)
print(f"  Student:  {student_speed:.2f}ms")

print("\n[TFLite 모델 속도 (마이크로초)]")
fp32_speed_us = evaluate_tflite_speed('models/base_fp32.tflite')
fp16_speed_us = evaluate_tflite_speed('models/base_fp16.tflite')
int8_speed_us = evaluate_tflite_speed('models/base_int8.tflite')
pruned_speed_us = evaluate_tflite_speed('models/pruned_model.tflite')
student_speed_us = evaluate_tflite_speed('models/student_fp16.tflite')

if fp32_speed_us:
    print(f"  FP32:    {fp32_speed_us:>8.2f}μs  (1.00x)")
    if fp16_speed_us:
        print(f"  FP16:    {fp16_speed_us:>8.2f}μs  ({fp32_speed_us/fp16_speed_us:.2f}x)")
    if int8_speed_us:
        print(f"  INT8:    {int8_speed_us:>8.2f}μs  ({fp32_speed_us/int8_speed_us:.2f}x)")
    if pruned_speed_us:
        print(f"  Pruned:  {pruned_speed_us:>8.2f}μs  ({fp32_speed_us/pruned_speed_us:.2f}x)")
    if student_speed_us:
        print(f"  Student: {student_speed_us:>8.2f}μs  ({fp32_speed_us/student_speed_us:.2f}x)")

# ============================================================================
# 8. 최종 종합 비교표
# ============================================================================
print("\n" + "="*80)
print("8. 📊 최종 종합 비교표")
print("="*80)

# 데이터 수집
teacher_params = count_parameters(teacher_model)
base_params = count_parameters(base_model)
pruned_params = count_parameters(final_pruned_model)
student_params = count_parameters(student_model)

teacher_size = os.path.getsize('models/teacher_model.h5') / 1024
base_size = os.path.getsize('models/base_model.h5') / 1024
pruned_tflite_size = len(tflite_pruned) / 1024
fp32_size = len(tflite_fp32) / 1024
fp16_size = len(tflite_fp16) / 1024
int8_size = len(tflite_int8) / 1024
student_size = len(tflite_student) / 1024

print("\n" + "┌" + "─"*118 + "┐")
print(f"│ {'모델':<25} {'정확도':<10} {'파라미터':<15} {'TF속도':<12} {'H5크기':<12} {'TFLite크기':<12} {'TFLite속도':<12} │")
print("├" + "─"*118 + "┤")

data = [
    ('Teacher (원본)', teacher_accuracy*100, teacher_params, teacher_speed, teacher_size, None, None),
    ('Base (중간)', base_accuracy*100, base_params, base_speed, base_size, fp32_size, fp32_speed_us),
    ('  + FP16 양자화', base_accuracy*100, base_params, None, None, fp16_size, fp16_speed_us),
    ('  + INT8 양자화', base_accuracy*100, base_params, None, None, int8_size, int8_speed_us),
    ('  + 프루닝 50%', pruned_accuracy*100, pruned_params, pruned_speed, None, pruned_tflite_size, pruned_speed_us),
    ('Student (단독)', scratch_accuracy*100, student_params, None, None, None, None),
    ('Student (증류)', distilled_accuracy*100, student_params, student_speed, None, student_size, student_speed_us),
]

for name, acc, params, tf_spd, h5_sz, tflite_sz, tflite_spd in data:
    acc_str = f"{acc:.2f}%"
    params_str = f"{params:,}개" if params else "-"
    tf_spd_str = f"{tf_spd:.2f}ms" if tf_spd else "-"
    h5_sz_str = f"{h5_sz:.1f}KB" if h5_sz else "-"
    tflite_sz_str = f"{tflite_sz:.1f}KB" if tflite_sz else "-"
    tflite_spd_str = f"{tflite_spd:.1f}μs" if tflite_spd else "-"
    
    print(f"│ {name:<25} {acc_str:<10} {params_str:<15} {tf_spd_str:<12} {h5_sz_str:<12} {tflite_sz_str:<12} {tflite_spd_str:<12} │")

print("└" + "─"*118 + "┘")

# ============================================================================
# 9. 시각화 및 인사이트
# ============================================================================
print("\n" + "="*80)
print("💡 핵심 인사이트")
print("="*80)

print("\n[1. 크기 감소 효과]")
max_size = teacher_size
print(f"  Teacher H5:      {teacher_size:>8.1f}KB  {'█'*50} (100%)")
print(f"  Base TFLite FP32:{fp32_size:>8.1f}KB  {'█'*int(fp32_size/max_size*50)} ({fp32_size/max_size*100:.1f}%)")
print(f"  Base TFLite FP16:{fp16_size:>8.1f}KB  {'█'*int(fp16_size/max_size*50)} ({fp16_size/max_size*100:.1f}%)")
print(f"  Base TFLite INT8:{int8_size:>8.1f}KB  {'█'*int(int8_size/max_size*50)} ({int8_size/max_size*100:.1f}%)")
print(f"  Pruned TFLite:   {pruned_tflite_size:>8.1f}KB  {'█'*int(pruned_tflite_size/max_size*50)} ({pruned_tflite_size/max_size*100:.1f}%)")
print(f"  Student TFLite:  {student_size:>8.1f}KB  {'█'*int(student_size/max_size*50)} ({student_size/max_size*100:.1f}%)")

if fp32_speed_us and int8_speed_us:
    print("\n[2. 속도 향상 효과 (TFLite 기준)]")
    print(f"  FP32:   {fp32_speed_us:>7.1f}μs  {'█'*50} (1.00x)")
    print(f"  FP16:   {fp16_speed_us:>7.1f}μs  {'█'*int(fp16_speed_us/fp32_speed_us*50)} ({fp32_speed_us/fp16_speed_us:.2f}x)")
    print(f"  INT8:   {int8_speed_us:>7.1f}μs  {'█'*int(int8_speed_us/fp32_speed_us*50)} ({fp32_speed_us/int8_speed_us:.2f}x)")
    print(f"  Pruned: {pruned_speed_us:>7.1f}μs  {'█'*int(pruned_speed_us/fp32_speed_us*50)} ({fp32_speed_us/pruned_speed_us:.2f}x)")
    print(f"  Student:{student_speed_us:>7.1f}μs  {'█'*int(student_speed_us/fp32_speed_us*50)} ({fp32_speed_us/student_speed_us:.2f}x)")

print("\n[3. 정확도 비교]")
for name, acc, *_ in data:
    bar_len = int(acc * 0.5)
    print(f"  {name:<25} {acc:>6.2f}%  {'█'*bar_len}")

print("\n[4. 파라미터 감소]")
print(f"  Teacher:   {teacher_params:>10,}개  (100.0%)")
print(f"  Base:      {base_params:>10,}개  ({base_params/teacher_params*100:>5.1f}%)")
print(f"  Student:   {student_params:>10,}개  ({student_params/teacher_params*100:>5.1f}%)")

print("\n[5. 경량화 기법별 요약]")
print(f"""
┌─ 양자화 (Quantization) ─────────────────────────────────┐
│ • FP32 → INT8: 크기 {((fp32_size-int8_size)/fp32_size*100):.1f}% 감소, 속도 {fp32_speed_us/int8_speed_us:.1f}x 향상            │
│ • 정확도 손실: 거의 없음 ({base_accuracy*100:.2f}%)                       │
│ • 추천: ⭐⭐⭐⭐⭐ (가장 먼저 시도)                           │
└─────────────────────────────────────────────────────────┘

┌─ 프루닝 (Pruning) ──────────────────────────────────────┐
│ • 50% 가중치 제거: {pruned_tflite_size:.1f}KB                               │
│ • 정확도: {pruned_accuracy*100:.2f}% ({(pruned_accuracy-base_accuracy)*100:+.2f}%p vs Base)                 │
│ • 추천: ⭐⭐⭐⭐ (양자화와 함께 사용)                         │
└─────────────────────────────────────────────────────────┘

┌─ 지식 증류 (Knowledge Distillation) ────────────────────┐
│ • 파라미터 {((teacher_params-student_params)/teacher_params*100):.1f}% 감소 (Teacher → Student)          │
│ • 증류 효과: {(distilled_accuracy-scratch_accuracy)*100:+.2f}%p 향상                          │
│ • 최종 정확도: {distilled_accuracy*100:.2f}% | 크기: {student_size:.1f}KB                   │
│ • 추천: ⭐⭐⭐ (극한 경량화 필요 시)                          │
└─────────────────────────────────────────────────────────┘
""")

print("\n[6. 실무 권장사항]")
print("""
🎯 시나리오별 최적 선택:

1️⃣ 균형잡힌 선택 (대부분의 경우)
   → Base 모델 + INT8 양자화
   → 크기: {:.1f}KB | 속도: {:.1f}μs | 정확도: {:.2f}%

2️⃣ 최고 정확도 유지
   → Base 모델 + FP16 양자화
   → 정확도: {:.2f}% | 크기: {:.1f}KB

3️⃣ 극한 경량화
   → Student 증류 + FP16
   → 크기: {:.1f}KB (Teacher의 {:.1f}%)

4️⃣ 최고 속도
   → Base + INT8 양자화
   → 속도: {:.1f}μs ({:.2f}x 향상)
""".format(
    int8_size, int8_speed_us, base_accuracy*100,
    base_accuracy*100, fp16_size,
    student_size, student_size/teacher_size*100,
    int8_speed_us, fp32_speed_us/int8_speed_us
))

print("\n" + "="*80)
print("✨ 모든 경량화 기법 비교 완료!")
print("="*80)
print(f"\n생성된 모델 파일:")
print(f"  models/teacher_model.h5")
print(f"  models/base_model.h5")
print(f"  models/pruned_model.h5")
print(f"  models/student_model.h5")
print(f"  models/base_fp32.tflite")
print(f"  models/base_fp16.tflite")
print(f"  models/base_int8.tflite")
print(f"  models/pruned_model.tflite")
print(f"  models/student_fp16.tflite")


TensorFlow 버전: 2.15.0
NumPy 버전: 1.24.3

1. MNIST 데이터 로드 및 전처리
학습 데이터: (55000, 28, 28, 1)
검증 데이터: (5000, 28, 28, 1)
테스트 데이터: (10000, 28, 28, 1)

2. Teacher 모델 (큰 모델) 생성 및 학습




[Teacher 모델 구조]
Model: "Teacher"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 64)        640       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 64)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 128)       73856     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 128)         0         
 g2D)                                                            
                                                               

c:\Users\yjcho\anaconda3\envs\mnist_final\lib\site-packages\tf_keras\src\engine\training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Epoch 1/10
430/430 [==============================] - 8s 15ms/step - loss: 0.2371 - accuracy: 0.9279 - val_loss: 0.0592 - val_accuracy: 0.9832
Epoch 2/10
430/430 [==============================] - 6s 14ms/step - loss: 0.0653 - accuracy: 0.9799 - val_loss: 0.0410 - val_accuracy: 0.9894
Epoch 3/10
430/430 [==============================] - 6s 14ms/step - loss: 0.0452 - accuracy: 0.9861 - val_loss: 0.0365 - val_accuracy: 0.9902
Epoch 4/10
430/430 [==============================] - 6s 14ms/step - loss: 0.0372 - accuracy: 0.9884 - val_loss: 0.0340 - val_accuracy: 0.9916
Epoch 5/10
430/430 [==============================] - 6s 14ms/step - loss: 0.0301 - accuracy: 0.9905 - val_loss: 0.0322 - val_accuracy: 0.9910
Epoch 6/10
430/430 [==============================] - 6s 14ms/step - loss: 0.0238 - accuracy: 0.9923 - val_loss: 0.0362 - val_accuracy: 0.9908
Epoch 7/10
430/430 [==============================] - 6s 14ms/step - loss: 0.0181 - accuracy: 0.9943 - val_loss: 0.0325 - val_accuracy: 0.9924

INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmp0tagrmuu\assets


✅ 프루닝 모델 TFLite: 225.39KB

5. 지식 증류 (Knowledge Distillation) 적용

[1단계] Student 모델 생성...

[Student 모델 구조]
Model: "Student"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 26, 26, 16)        160       
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 13, 13, 16)        0         
 g2D)                                                            
                                                                 
 conv2d_7 (Conv2D)           (None, 11, 11, 32)        4640      
                                                                 
 max_pooling2d_7 (MaxPoolin  (None, 5, 5, 32)          0         
 g2D)                                                            
                                                                 
 flatten_3 (Flatten)         (None, 800)               0         
                    

...

  FP16: 443.81KB (49.7% 감소)
INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmph61hm4bd\assets


INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmph61hm4bd\assets
c:\Users\yjcho\anaconda3\envs\mnist_final\lib\site-packages\tensorflow\lite\python\convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


  INT8: 226.78KB (74.3% 감소)

[Student 모델 양자화]
INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmp7_ub5zqa\assets


INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmp7_ub5zqa\assets


  Student FP16: 115.13KB

7. 성능 측정

[TensorFlow 모델 속도 (밀리초)]
  Teacher:  31.33ms
  Base:     31.03ms
  Pruned:   31.31ms
  Student:  30.78ms

[TFLite 모델 속도 (마이크로초)]
  FP32:       81.27μs  (1.00x)
  FP16:       69.82μs  (1.16x)
  INT8:       67.95μs  (1.20x)
  Pruned:    204.24μs  (0.40x)
  Student:    29.24μs  (2.78x)

8. 📊 최종 종합 비교표

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ 모델                        정확도        파라미터            TF속도         H5크기         TFLite크기     TFLite속도     │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Teacher (원본)              99.19%     896,522개        31.33ms      10550.4KB    -            -            │
│ Base (중간)                 99.08%     225,034개        31.03ms      2682.1KB     882.2KB      81.3μs       │
│   + FP16 양자화              99.08%     225,034개        -            -            443.8KB      

In [1]:
"""
CIFAR-10 온디바이스 AI 경량화 기법 종합 비교
- 양자화 (Quantization): FP32 → FP16 → INT8
- 프루닝 (Pruning): 50% 가중치 제거
- 지식 증류 (Knowledge Distillation): Teacher → Student
"""

import tensorflow as tf
from tensorflow import keras
import tensorflow_model_optimization as tfmot
import numpy as np
import time
import os
from pathlib import Path

print(f"TensorFlow 버전: {tf.__version__}")
print(f"NumPy 버전: {np.__version__}")

# 디렉토리 생성
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

# ============================================================================
# 1. 데이터 로드 및 전처리
# ============================================================================
print("\n" + "="*80)
print("1. CIFAR-10 데이터 로드 및 전처리")
print("="*80)

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# 정규화 (CIFAR-10은 32x32x3 컬러 이미지)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# y를 1D로 변환
y_train = y_train.flatten()
y_test = y_test.flatten()

# 검증 데이터 분리
x_val = x_train[-5000:]
y_val = y_train[-5000:]
x_train = x_train[:-5000]
y_train = y_train[:-5000]

print(f"학습 데이터: {x_train.shape}")
print(f"검증 데이터: {x_val.shape}")
print(f"테스트 데이터: {x_test.shape}")
print(f"클래스: 10개 (비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭)")

# ============================================================================
# 2. Teacher 모델 (큰 모델) 생성 및 학습
# ============================================================================
print("\n" + "="*80)
print("2. Teacher 모델 (큰 모델) 생성 및 학습")
print("="*80)

def create_teacher_model():
    """Teacher 모델 - ~2M 파라미터"""
    model = keras.Sequential([
        keras.layers.Conv2D(64, 3, padding='same', activation='relu', input_shape=(32, 32, 3)),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Dropout(0.2),
        
        keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Dropout(0.3),
        
        keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Dropout(0.4),
        
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(10, activation='softmax')
    ], name='Teacher')
    return model

teacher_model = create_teacher_model()
teacher_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n[Teacher 모델 구조]")
teacher_model.summary()

print("\nTeacher 모델 학습 시작...")
teacher_model.fit(
    x_train, y_train,
    epochs=20,  # CIFAR-10은 더 많은 epoch 필요
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=1
)

teacher_loss, teacher_accuracy = teacher_model.evaluate(x_test, y_test, verbose=0)
print(f"\n✅ Teacher 정확도: {teacher_accuracy*100:.2f}%")

teacher_model.save('models/teacher_model.h5')

# ============================================================================
# 3. 기본 모델 (중간 크기) 생성 및 학습
# ============================================================================
print("\n" + "="*80)
print("3. 기본 모델 (중간 크기) 생성 및 학습")
print("="*80)

def create_base_model():
    """기본 모델 - ~500K 파라미터"""
    model = keras.Sequential([
        keras.layers.Conv2D(32, 3, padding='same', activation='relu', input_shape=(32, 32, 3)),
        keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Dropout(0.2),
        
        keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Dropout(0.3),
        
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(10, activation='softmax')
    ], name='Base')
    return model

base_model = create_base_model()
base_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n[기본 모델 구조]")
base_model.summary()

print("\n기본 모델 학습 시작...")
base_model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=1
)

base_loss, base_accuracy = base_model.evaluate(x_test, y_test, verbose=0)
print(f"\n✅ 기본 모델 정확도: {base_accuracy*100:.2f}%")

base_model.save('models/base_model.h5')

# ============================================================================
# 4. 프루닝 (Pruning) 적용
# ============================================================================
print("\n" + "="*80)
print("4. 프루닝 (Pruning) 적용")
print("="*80)

print("\n[1단계] 프루닝용 모델 사전 학습...")
pruning_model = create_base_model()
pruning_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

pruning_model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=0
)

pre_prune_loss, pre_prune_accuracy = pruning_model.evaluate(x_test, y_test, verbose=0)
print(f"✅ 프루닝 전 정확도: {pre_prune_accuracy*100:.2f}%")

pruning_model.save('models/base_before_pruning.h5')
base_h5_size = os.path.getsize('models/base_before_pruning.h5') / 1024
print(f"프루닝 전 H5 크기: {base_h5_size:.2f}KB")

print("\n[2단계] 프루닝 적용 (50% 가중치 제거)...")
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=3000
    )
}

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(pruning_model, **pruning_params)
pruned_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n[3단계] 프루닝 모델 파인튜닝...")
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=0
)

pruned_loss, pruned_accuracy = pruned_model.evaluate(x_test, y_test, verbose=0)
print(f"✅ 프루닝 후 정확도: {pruned_accuracy*100:.2f}%")

print("\n[4단계] 프루닝 확정...")
final_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
final_pruned_model.save('models/pruned_model.h5')

# 프루닝 모델 TFLite 변환
converter_pruned = tf.lite.TFLiteConverter.from_keras_model(final_pruned_model)
converter_pruned.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_pruned = converter_pruned.convert()
with open('models/pruned_model.tflite', 'wb') as f:
    f.write(tflite_pruned)
print(f"✅ 프루닝 모델 TFLite: {len(tflite_pruned)/1024:.2f}KB")

# ============================================================================
# 5. 지식 증류 (Knowledge Distillation)
# ============================================================================
print("\n" + "="*80)
print("5. 지식 증류 (Knowledge Distillation) 적용")
print("="*80)

class DistillationModel(keras.Model):
    """지식 증류를 위한 커스텀 모델"""
    def __init__(self, teacher, student):
        super().__init__()
        self.teacher = teacher
        self.student = student
        
    def compile(self, optimizer, metrics, student_loss_fn, distillation_loss_fn, alpha=0.1, temperature=3):
        super().compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn
        self.alpha = alpha
        self.temperature = temperature
        
    def train_step(self, data):
        x, y = data
        teacher_predictions = self.teacher(x, training=False)
        
        with tf.GradientTape() as tape:
            student_predictions = self.student(x, training=True)
            student_loss = self.student_loss_fn(y, student_predictions)
            distillation_loss = self.distillation_loss_fn(
                tf.nn.softmax(teacher_predictions / self.temperature, axis=1),
                tf.nn.softmax(student_predictions / self.temperature, axis=1)
            ) * (self.temperature ** 2)
            total_loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss
            
        trainable_vars = self.student.trainable_variables
        gradients = tape.gradient(total_loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        self.compiled_metrics.update_state(y, student_predictions)
        
        results = {m.name: m.result() for m in self.metrics}
        results.update({
            "student_loss": student_loss,
            "distillation_loss": distillation_loss,
            "total_loss": total_loss
        })
        return results
    
    def test_step(self, data):
        x, y = data
        y_prediction = self.student(x, training=False)
        self.compiled_metrics.update_state(y, y_prediction)
        return {m.name: m.result() for m in self.metrics}

def create_student_model():
    """Student 모델 - ~100K 파라미터"""
    model = keras.Sequential([
        keras.layers.Conv2D(16, 3, padding='same', activation='relu', input_shape=(32, 32, 3)),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ], name='Student')
    return model

print("\n[1단계] Student 모델 생성...")
student_model = create_student_model()
print("\n[Student 모델 구조]")
student_model.summary()

print("\n[2단계] Student 단독 학습 (비교용)...")
student_scratch = create_student_model()
student_scratch.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

student_scratch.fit(
    x_train, y_train,
    epochs=20,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=0
)

scratch_loss, scratch_accuracy = student_scratch.evaluate(x_test, y_test, verbose=0)
print(f"✅ Student 단독 학습 정확도: {scratch_accuracy*100:.2f}%")

print("\n[3단계] 지식 증류로 Student 학습...")
distillation_model = DistillationModel(teacher=teacher_model, student=student_model)
distillation_model.compile(
    optimizer=keras.optimizers.Adam(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
    student_loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    distillation_loss_fn=keras.losses.KLDivergence(),
    alpha=0.1,
    temperature=3
)

distillation_model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=0
)

# Student 모델로 직접 평가
student_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
distilled_loss, distilled_accuracy = student_model.evaluate(x_test, y_test, verbose=0)
print(f"✅ 지식 증류 후 정확도: {distilled_accuracy*100:.2f}%")
print(f"💡 증류 효과: {(distilled_accuracy - scratch_accuracy)*100:+.2f}%p")

student_model.save('models/student_model.h5')

# ============================================================================
# 6. 양자화 (Quantization) 변환
# ============================================================================
print("\n" + "="*80)
print("6. 양자화 (Quantization) 변환")
print("="*80)

print("\n[기본 모델 양자화]")

# FP32 TFLite
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(base_model)
tflite_fp32 = converter_fp32.convert()
with open('models/base_fp32.tflite', 'wb') as f:
    f.write(tflite_fp32)
print(f"  FP32: {len(tflite_fp32)/1024:.2f}KB")

# FP16 TFLite
converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(base_model)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter_fp16.convert()
with open('models/base_fp16.tflite', 'wb') as f:
    f.write(tflite_fp16)
print(f"  FP16: {len(tflite_fp16)/1024:.2f}KB ({((len(tflite_fp32)-len(tflite_fp16))/len(tflite_fp32)*100):.1f}% 감소)")

# INT8 TFLite
def representative_dataset():
    for i in range(100):
        yield [x_train[i:i+1]]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(base_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8
tflite_int8 = converter_int8.convert()
with open('models/base_int8.tflite', 'wb') as f:
    f.write(tflite_int8)
print(f"  INT8: {len(tflite_int8)/1024:.2f}KB ({((len(tflite_fp32)-len(tflite_int8))/len(tflite_fp32)*100):.1f}% 감소)")

print("\n[Student 모델 양자화]")
converter_student = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter_student.optimizations = [tf.lite.Optimize.DEFAULT]
converter_student.target_spec.supported_types = [tf.float16]
tflite_student = converter_student.convert()
with open('models/student_fp16.tflite', 'wb') as f:
    f.write(tflite_student)
print(f"  Student FP16: {len(tflite_student)/1024:.2f}KB")

# ============================================================================
# 7. 성능 측정 함수
# ============================================================================
print("\n" + "="*80)
print("7. 성능 측정")
print("="*80)

def count_parameters(model):
    """모델 파라미터 수 계산"""
    return sum([np.prod(var.shape) for var in model.trainable_variables])

def evaluate_tf_model_speed(model, iterations=100):
    """TensorFlow 모델 속도 측정"""
    # 워밍업
    for _ in range(10):
        model.predict(x_test[:1], verbose=0)
    
    # 측정
    start = time.time()
    for _ in range(iterations):
        model.predict(x_test[:1], verbose=0)
    end = time.time()
    
    return (end - start) / iterations * 1000

def evaluate_tflite_speed(tflite_path, iterations=1000):
    """TFLite 모델 속도 측정 (마이크로초)"""
    if not os.path.exists(tflite_path):
        return None
    
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    input_dtype = input_details[0]['dtype']
    
    # 다양한 입력 준비
    test_images = []
    for i in range(min(100, len(x_test))):
        test_image = x_test[i:i+1]
        if input_dtype == np.uint8:
            input_scale, input_zero_point = input_details[0]['quantization']
            test_image = test_image / input_scale + input_zero_point
            test_image = test_image.astype(np.uint8)
        test_images.append(test_image)
    
    # 워밍업
    for i in range(20):
        interpreter.set_tensor(input_details[0]['index'], test_images[i % len(test_images)])
        interpreter.invoke()
    
    # 측정
    start = time.time()
    for i in range(iterations):
        interpreter.set_tensor(input_details[0]['index'], test_images[i % len(test_images)])
        interpreter.invoke()
    end = time.time()
    
    return (end - start) / iterations * 1000000  # μs

print("\n[TensorFlow 모델 속도 (밀리초)]")
teacher_speed = evaluate_tf_model_speed(teacher_model)
print(f"  Teacher:  {teacher_speed:.2f}ms")

base_speed = evaluate_tf_model_speed(base_model)
print(f"  Base:     {base_speed:.2f}ms")

pruned_speed = evaluate_tf_model_speed(final_pruned_model)
print(f"  Pruned:   {pruned_speed:.2f}ms")

student_speed = evaluate_tf_model_speed(student_model)
print(f"  Student:  {student_speed:.2f}ms")

print("\n[TFLite 모델 속도 (마이크로초)]")
fp32_speed_us = evaluate_tflite_speed('models/base_fp32.tflite')
fp16_speed_us = evaluate_tflite_speed('models/base_fp16.tflite')
int8_speed_us = evaluate_tflite_speed('models/base_int8.tflite')
pruned_speed_us = evaluate_tflite_speed('models/pruned_model.tflite')
student_speed_us = evaluate_tflite_speed('models/student_fp16.tflite')

if fp32_speed_us:
    print(f"  FP32:    {fp32_speed_us:>8.2f}μs  (1.00x)")
    if fp16_speed_us:
        print(f"  FP16:    {fp16_speed_us:>8.2f}μs  ({fp32_speed_us/fp16_speed_us:.2f}x)")
    if int8_speed_us:
        print(f"  INT8:    {int8_speed_us:>8.2f}μs  ({fp32_speed_us/int8_speed_us:.2f}x)")
    if pruned_speed_us:
        print(f"  Pruned:  {pruned_speed_us:>8.2f}μs  ({fp32_speed_us/pruned_speed_us:.2f}x)")
    if student_speed_us:
        print(f"  Student: {student_speed_us:>8.2f}μs  ({fp32_speed_us/student_speed_us:.2f}x)")

# ============================================================================
# 8. 최종 종합 비교표
# ============================================================================
print("\n" + "="*80)
print("8. 📊 최종 종합 비교표")
print("="*80)

# 데이터 수집
teacher_params = count_parameters(teacher_model)
base_params = count_parameters(base_model)
pruned_params = count_parameters(final_pruned_model)
student_params = count_parameters(student_model)

teacher_size = os.path.getsize('models/teacher_model.h5') / 1024
base_size = os.path.getsize('models/base_model.h5') / 1024
pruned_tflite_size = len(tflite_pruned) / 1024
fp32_size = len(tflite_fp32) / 1024
fp16_size = len(tflite_fp16) / 1024
int8_size = len(tflite_int8) / 1024
student_size = len(tflite_student) / 1024

print("\n" + "┌" + "─"*118 + "┐")
print(f"│ {'모델':<25} {'정확도':<10} {'파라미터':<15} {'TF속도':<12} {'H5크기':<12} {'TFLite크기':<12} {'TFLite속도':<12} │")
print("├" + "─"*118 + "┤")

data = [
    ('Teacher (원본)', teacher_accuracy*100, teacher_params, teacher_speed, teacher_size, None, None),
    ('Base (중간)', base_accuracy*100, base_params, base_speed, base_size, fp32_size, fp32_speed_us),
    ('  + FP16 양자화', base_accuracy*100, base_params, None, None, fp16_size, fp16_speed_us),
    ('  + INT8 양자화', base_accuracy*100, base_params, None, None, int8_size, int8_speed_us),
    ('  + 프루닝 50%', pruned_accuracy*100, pruned_params, pruned_speed, None, pruned_tflite_size, pruned_speed_us),
    ('Student (단독)', scratch_accuracy*100, student_params, None, None, None, None),
    ('Student (증류)', distilled_accuracy*100, student_params, student_speed, None, student_size, student_speed_us),
]

for name, acc, params, tf_spd, h5_sz, tflite_sz, tflite_spd in data:
    acc_str = f"{acc:.2f}%"
    params_str = f"{params:,}개" if params else "-"
    tf_spd_str = f"{tf_spd:.2f}ms" if tf_spd else "-"
    h5_sz_str = f"{h5_sz:.1f}KB" if h5_sz else "-"
    tflite_sz_str = f"{tflite_sz:.1f}KB" if tflite_sz else "-"
    tflite_spd_str = f"{tflite_spd:.1f}μs" if tflite_spd else "-"
    
    print(f"│ {name:<25} {acc_str:<10} {params_str:<15} {tf_spd_str:<12} {h5_sz_str:<12} {tflite_sz_str:<12} {tflite_spd_str:<12} │")

print("└" + "─"*118 + "┘")

# ============================================================================
# 9. 시각화 및 인사이트
# ============================================================================
print("\n" + "="*80)
print("💡 핵심 인사이트")
print("="*80)

print("\n[1. 크기 감소 효과]")
max_size = teacher_size
print(f"  Teacher H5:      {teacher_size:>8.1f}KB  {'█'*50} (100%)")
print(f"  Base TFLite FP32:{fp32_size:>8.1f}KB  {'█'*int(fp32_size/max_size*50)} ({fp32_size/max_size*100:.1f}%)")
print(f"  Base TFLite FP16:{fp16_size:>8.1f}KB  {'█'*int(fp16_size/max_size*50)} ({fp16_size/max_size*100:.1f}%)")
print(f"  Base TFLite INT8:{int8_size:>8.1f}KB  {'█'*int(int8_size/max_size*50)} ({int8_size/max_size*100:.1f}%)")
print(f"  Pruned TFLite:   {pruned_tflite_size:>8.1f}KB  {'█'*int(pruned_tflite_size/max_size*50)} ({pruned_tflite_size/max_size*100:.1f}%)")
print(f"  Student TFLite:  {student_size:>8.1f}KB  {'█'*int(student_size/max_size*50)} ({student_size/max_size*100:.1f}%)")

if fp32_speed_us and int8_speed_us:
    print("\n[2. 속도 향상 효과 (TFLite 기준)]")
    print(f"  FP32:   {fp32_speed_us:>7.1f}μs  {'█'*50} (1.00x)")
    print(f"  FP16:   {fp16_speed_us:>7.1f}μs  {'█'*int(fp16_speed_us/fp32_speed_us*50)} ({fp32_speed_us/fp16_speed_us:.2f}x)")
    print(f"  INT8:   {int8_speed_us:>7.1f}μs  {'█'*int(int8_speed_us/fp32_speed_us*50)} ({fp32_speed_us/int8_speed_us:.2f}x)")
    print(f"  Pruned: {pruned_speed_us:>7.1f}μs  {'█'*int(pruned_speed_us/fp32_speed_us*50)} ({fp32_speed_us/pruned_speed_us:.2f}x)")
    print(f"  Student:{student_speed_us:>7.1f}μs  {'█'*int(student_speed_us/fp32_speed_us*50)} ({fp32_speed_us/student_speed_us:.2f}x)")

print("\n[3. 정확도 비교]")
for name, acc, *_ in data:
    bar_len = int(acc * 0.5)
    print(f"  {name:<25} {acc:>6.2f}%  {'█'*bar_len}")

print("\n[4. 파라미터 감소]")
print(f"  Teacher:   {teacher_params:>10,}개  (100.0%)")
print(f"  Base:      {base_params:>10,}개  ({base_params/teacher_params*100:>5.1f}%)")
print(f"  Student:   {student_params:>10,}개  ({student_params/teacher_params*100:>5.1f}%)")

print("\n[5. 경량화 기법별 요약]")
print(f"""
┌─ 양자화 (Quantization) ─────────────────────────────────┐
│ • FP32 → INT8: 크기 {((fp32_size-int8_size)/fp32_size*100):.1f}% 감소, 속도 {fp32_speed_us/int8_speed_us:.1f}x 향상            │
│ • 정확도 손실: {(base_accuracy-base_accuracy)*100:.2f}%p (양자화는 정확도 측정 별도 필요)    │
│ • 추천: ⭐⭐⭐⭐⭐ (가장 먼저 시도)                           │
└─────────────────────────────────────────────────────────┘

┌─ 프루닝 (Pruning) ──────────────────────────────────────┐
│ • 50% 가중치 제거: {pruned_tflite_size:.1f}KB                               │
│ • 정확도: {pruned_accuracy*100:.2f}% ({(pruned_accuracy-base_accuracy)*100:+.2f}%p vs Base)                 │
│ • 추천: ⭐⭐⭐⭐ (양자화와 함께 사용)                         │
└─────────────────────────────────────────────────────────┘

┌─ 지식 증류 (Knowledge Distillation) ────────────────────┐
│ • 파라미터 {((teacher_params-student_params)/teacher_params*100):.1f}% 감소 (Teacher → Student)          │
│ • 증류 효과: {(distilled_accuracy-scratch_accuracy)*100:+.2f}%p 향상                          │
│ • 최종 정확도: {distilled_accuracy*100:.2f}% | 크기: {student_size:.1f}KB                   │
│ • 추천: ⭐⭐⭐ (극한 경량화 필요 시)                          │
└─────────────────────────────────────────────────────────┘
""")

print("\n[6. 실무 권장사항]")
print("""
🎯 시나리오별 최적 선택:

1️⃣ 균형잡힌 선택 (대부분의 경우)
   → Base 모델 + INT8 양자화
   → 크기: {:.1f}KB | 속도: {:.1f}μs | 정확도: {:.2f}%

2️⃣ 최고 정확도 유지
   → Base 모델 + FP16 양자화
   → 정확도: {:.2f}% | 크기: {:.1f}KB

3️⃣ 극한 경량화
   → Student 증류 + FP16
   → 크기: {:.1f}KB (Teacher의 {:.1f}%)

4️⃣ 최고 속도
   → Base + INT8 양자화
   → 속도: {:.1f}μs ({:.2f}x 향상)
""".format(
    int8_size, int8_speed_us, base_accuracy*100,
    base_accuracy*100, fp16_size,
    student_size, student_size/teacher_size*100,
    int8_speed_us, fp32_speed_us/int8_speed_us
))

print("\n" + "="*80)
print("✨ 모든 경량화 기법 비교 완료!")
print("="*80)
print(f"\n생성된 모델 파일:")
print(f"  models/teacher_model.h5")
print(f"  models/base_model.h5")
print(f"  models/pruned_model.h5")
print(f"  models/student_model.h5")
print(f"  models/base_fp32.tflite")
print(f"  models/base_fp16.tflite")
print(f"  models/base_int8.tflite")
print(f"  models/pruned_model.tflite")
print(f"  models/student_fp16.tflite")


TensorFlow 버전: 2.15.0
NumPy 버전: 1.24.3

1. CIFAR-10 데이터 로드 및 전처리
학습 데이터: (45000, 32, 32, 3)
검증 데이터: (5000, 32, 32, 3)
테스트 데이터: (10000, 32, 32, 3)
클래스: 10개 (비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭)

2. Teacher 모델 (큰 모델) 생성 및 학습




[Teacher 모델 구조]
Model: "Teacher"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 64)        1792      
                                                                 
 batch_normalization (Batch  (None, 32, 32, 64)        256       
 Normalization)                                                  
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 32, 64)        36928     
                                                                 
 max_pooling2d (MaxPooling2  (None, 16, 16, 64)        0         
 D)                                                              
          

c:\Users\yjcho\anaconda3\envs\mnist_final\lib\site-packages\tf_keras\src\engine\training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Epoch 1/20
352/352 [==============================] - 15s 41ms/step - loss: 1.6392 - accuracy: 0.4028 - val_loss: 1.2435 - val_accuracy: 0.5582
Epoch 2/20
352/352 [==============================] - 14s 40ms/step - loss: 1.2271 - accuracy: 0.5610 - val_loss: 0.9987 - val_accuracy: 0.6506
Epoch 3/20
352/352 [==============================] - 14s 40ms/step - loss: 1.0439 - accuracy: 0.6310 - val_loss: 0.8718 - val_accuracy: 0.6978
Epoch 4/20
352/352 [==============================] - 14s 40ms/step - loss: 0.9373 - accuracy: 0.6727 - val_loss: 0.8096 - val_accuracy: 0.7222
Epoch 5/20
352/352 [==============================] - 14s 40ms/step - loss: 0.8524 - accuracy: 0.6969 - val_loss: 0.8256 - val_accuracy: 0.7138
Epoch 6/20
352/352 [==============================] - 14s 41ms/step - loss: 0.7867 - accuracy: 0.7219 - val_loss: 0.7076 - val_accuracy: 0.7560
Epoch 7/20
352/352 [==============================] - 14s 39ms/step - loss: 0.7258 - accuracy: 0.7456 - val_loss: 0.7320 - val_accuracy:

INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmp9yzv5syy\assets


✅ 프루닝 모델 TFLite: 1100.86KB

5. 지식 증류 (Knowledge Distillation) 적용

[1단계] Student 모델 생성...

[Student 모델 구조]
Model: "Student"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_14 (Conv2D)          (None, 32, 32, 16)        448       
                                                                 
 max_pooling2d_7 (MaxPoolin  (None, 16, 16, 16)        0         
 g2D)                                                            
                                                                 
 conv2d_15 (Conv2D)          (None, 16, 16, 32)        4640      
                                                                 
 max_pooling2d_8 (MaxPoolin  (None, 8, 8, 32)          0         
 g2D)                                                            
                                                                 
 flatten_3 (Flatten)         (None, 2048)              0         
                   

...

  FP16: 2187.28KB (49.9% 감소)
INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmpwyjhvk_m\assets


INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmpwyjhvk_m\assets
c:\Users\yjcho\anaconda3\envs\mnist_final\lib\site-packages\tensorflow\lite\python\convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


  INT8: 1101.79KB (74.8% 감소)

[Student 모델 양자화]
INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmpdahrvhu1\assets


INFO:tensorflow:Assets written to: C:\Users\yjcho\AppData\Local\Temp\tmpdahrvhu1\assets


  Student FP16: 529.08KB

7. 성능 측정

[TensorFlow 모델 속도 (밀리초)]
  Teacher:  35.47ms
  Base:     32.18ms
  Pruned:   32.35ms
  Student:  31.34ms

[TFLite 모델 속도 (마이크로초)]
  FP32:      672.40μs  (1.00x)
  FP16:      682.68μs  (0.98x)
  INT8:      365.67μs  (1.84x)
  Pruned:   1806.27μs  (0.37x)
  Student:    55.90μs  (12.03x)

8. 📊 최종 종합 비교표

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ 모델                        정확도        파라미터            TF속도         H5크기         TFLite크기     TFLite속도     │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Teacher (원본)              83.03%     3,249,098개      35.47ms      38180.4KB    -            -            │
│ Base (중간)                 78.62%     1,116,970개      32.18ms      13147.6KB    4367.3KB     672.4μs      │
│   + FP16 양자화              78.62%     1,116,970개      -            -            2187.3KB    

In [ ]:
"""
CIFAR-10 온디바이스 AI 테스트 - 올인원 스크립트

1. 테스트 데이터 바이너리 생성 (Android용)
2. Python에서 직접 테스트
3. 결과 비교를 위한 기준값 제공

Android 앱 실행 후 결과와 비교하세요!
"""

import tensorflow as tf
import numpy as np
import struct
import time
import os
from pathlib import Path

# ============================================================================
# 설정
# ============================================================================
TOTAL_TEST_COUNT = 1000  # 테스트 개수 (1000 또는 10000)
OUTPUT_FILENAME = f"cifar10_test_{TOTAL_TEST_COUNT}.bin"
MODEL_PATH = "models/base_int8.tflite"

CLASSES = ["비행기", "자동차", "새", "고양이", "사슴", "개", "개구리", "말", "배", "트럭"]

print("="*80)
print(f"🚀 CIFAR-10 온디바이스 AI 테스트 준비 ({TOTAL_TEST_COUNT}개)")
print("="*80)

# ============================================================================
# 1. 테스트 데이터 바이너리 생성
# ============================================================================
print("\n" + "="*80)
print("📦 1단계: 테스트 데이터 바이너리 생성")
print("="*80)

# CIFAR-10 로드
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# 정규화
x_test = x_test.astype('float32') / 255.0
y_test = y_test.flatten()

# 필요한 개수만 추출
x_test_subset = x_test[:TOTAL_TEST_COUNT]
y_test_subset = y_test[:TOTAL_TEST_COUNT]

print(f"\n데이터 정보:")
print(f"  Shape: {x_test_subset.shape}")
print(f"  레이블 범위: {y_test_subset.min()} ~ {y_test_subset.max()}")
print(f"  픽셀 범위: {x_test_subset.min():.3f} ~ {x_test_subset.max():.3f}")

# 바이너리 파일 생성
print(f"\n바이너리 파일 생성 중: {OUTPUT_FILENAME}")

with open(OUTPUT_FILENAME, 'wb') as f:
    for i in range(TOTAL_TEST_COUNT):
        # Label (1 byte, unsigned)
        f.write(struct.pack('B', y_test_subset[i]))
        
        # Pixels (32*32*3 floats, LITTLE ENDIAN)
        pixels = x_test_subset[i].flatten()
        for pixel in pixels:
            f.write(struct.pack('<f', pixel))
        
        # 진행상황 표시
        if (i + 1) % 100 == 0 or i == 0:
            progress = (i + 1) / TOTAL_TEST_COUNT * 100
            print(f"  진행: {i + 1}/{TOTAL_TEST_COUNT} ({progress:.1f}%)")

file_size_mb = os.path.getsize(OUTPUT_FILENAME) / 1024 / 1024
print(f"\n✅ 생성 완료!")
print(f"  파일: {OUTPUT_FILENAME}")
print(f"  크기: {file_size_mb:.2f} MB")

# 검증
print(f"\n🔍 데이터 검증 (처음 3개):")
with open(OUTPUT_FILENAME, 'rb') as f:
    for idx in range(min(3, TOTAL_TEST_COUNT)):
        label = struct.unpack('B', f.read(1))[0]
        pixels = []
        for _ in range(3072):
            pixel = struct.unpack('<f', f.read(4))[0]
            pixels.append(pixel)
        pixels = np.array(pixels).reshape(32, 32, 3)
        
        is_correct = np.allclose(pixels, x_test_subset[idx])
        status = "✅" if is_correct else "❌"
        
        print(f"  이미지 #{idx}: 레이블={label} ({CLASSES[label]}), "
              f"범위=[{pixels.min():.3f}, {pixels.max():.3f}] {status}")

# ============================================================================
# 2. Python에서 모델 테스트
# ============================================================================
print("\n" + "="*80)
print("🧪 2단계: Python에서 모델 테스트")
print("="*80)

# 모델 존재 확인
if not os.path.exists(MODEL_PATH):
    print(f"\n❌ 오류: '{MODEL_PATH}' 파일이 없습니다!")
    print("먼저 모델을 생성하고 이 폴더에 복사하세요.")
    exit(1)

# 모델 로드
print(f"\n모델 로드: {MODEL_PATH}")
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\n모델 정보:")
print(f"  Input shape: {input_details[0]['shape']}")
print(f"  Input type: {input_details[0]['dtype']}")
print(f"  Output shape: {output_details[0]['shape']}")
print(f"  Output type: {output_details[0]['dtype']}")

# 추론 시작
print(f"\n추론 시작 ({TOTAL_TEST_COUNT}개)...")

correct = 0
inference_times_us = []
predictions = []

for i in range(TOTAL_TEST_COUNT):
    # 입력 전처리: Float (0~1) -> UINT8 (0~255)
    input_data = (x_test_subset[i:i+1] * 255).astype(np.uint8)
    
    # 추론 시간 측정
    start = time.time()
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    inference_time_us = (time.time() - start) * 1000000
    inference_times_us.append(inference_time_us)
    
    # 출력
    output_data = interpreter.get_tensor(output_details[0]['index'])[0]
    
    # Softmax
    logits = output_data.astype(np.float32) / 255.0
    exp_logits = np.exp(logits - np.max(logits))
    probabilities = exp_logits / exp_logits.sum()
    
    predicted_class = np.argmax(probabilities)
    confidence = probabilities[predicted_class]
    actual_class = y_test_subset[i]
    
    # 정확도 계산
    is_correct = (predicted_class == actual_class)
    if is_correct:
        correct += 1
    
    predictions.append({
        'index': i,
        'actual': actual_class,
        'predicted': predicted_class,
        'confidence': confidence,
        'correct': is_correct,
        'time_us': inference_time_us
    })
    
    # 진행상황 표시
    if (i + 1) % 100 == 0:
        current_acc = correct / (i + 1) * 100
        avg_time = np.mean(inference_times_us) / 1000
        print(f"  진행: {i + 1}/{TOTAL_TEST_COUNT} - "
              f"정확도: {current_acc:.2f}% - "
              f"평균 시간: {avg_time:.2f}ms")

# ============================================================================
# 3. 결과 분석
# ============================================================================
print("\n" + "="*80)
print("📊 3단계: 결과 분석")
print("="*80)

accuracy = correct / TOTAL_TEST_COUNT
avg_time_us = np.mean(inference_times_us)
avg_time_ms = avg_time_us / 1000
std_time_ms = np.std(inference_times_us) / 1000

print(f"\n[Python 테스트 결과]")
print(f"  정확도:       {accuracy * 100:.2f}% ({correct}/{TOTAL_TEST_COUNT})")
print(f"  평균 시간:    {avg_time_ms:.2f}ms ({avg_time_us:.0f}μs)")
print(f"  표준편차:     {std_time_ms:.2f}ms")
print(f"  최소 시간:    {np.min(inference_times_us) / 1000:.2f}ms")
print(f"  최대 시간:    {np.max(inference_times_us) / 1000:.2f}ms")

# 신뢰구간 계산 (샘플이 충분할 때만)
if TOTAL_TEST_COUNT >= 100:
    stderr = np.sqrt(accuracy * (1 - accuracy) / TOTAL_TEST_COUNT)
    margin = 1.96 * stderr  # 95% 신뢰구간
    ci_lower = (accuracy - margin) * 100
    ci_upper = (accuracy + margin) * 100
    
    print(f"\n[통계적 신뢰도]")
    print(f"  95% 신뢰구간: {ci_lower:.2f}% ~ {ci_upper:.2f}%")
    print(f"  오차범위:     ±{margin * 100:.2f}%p")

# 클래스별 정확도
print(f"\n[클래스별 정확도]")
class_correct = [0] * 10
class_total = [0] * 10

for pred in predictions:
    actual = pred['actual']
    class_total[actual] += 1
    if pred['correct']:
        class_correct[actual] += 1

for i in range(10):
    if class_total[i] > 0:
        class_acc = class_correct[i] / class_total[i] * 100
        bar_length = int(class_acc / 2)
        bar = "█" * bar_length
        print(f"  {CLASSES[i]:6s}: {class_acc:5.1f}% ({class_correct[i]:3d}/{class_total[i]:3d}) {bar}")
    else:
        print(f"  {CLASSES[i]:6s}:   N/A")

# 틀린 예측 상위 10개
print(f"\n[가장 확신있게 틀린 예측 TOP 10]")
wrong_predictions = [p for p in predictions if not p['correct']]
wrong_predictions.sort(key=lambda x: x['confidence'], reverse=True)

for i, pred in enumerate(wrong_predictions[:10]):
    print(f"  {i+1}. 이미지 #{pred['index']:4d}: "
          f"{CLASSES[pred['actual']]:6s} → {CLASSES[pred['predicted']]:6s} "
          f"(확신도: {pred['confidence']:.1%})")

# 맞은 예측 상위 5개
print(f"\n[가장 확신있게 맞춘 예측 TOP 5]")
correct_predictions = [p for p in predictions if p['correct']]
correct_predictions.sort(key=lambda x: x['confidence'], reverse=True)

for i, pred in enumerate(correct_predictions[:5]):
    print(f"  {i+1}. 이미지 #{pred['index']:4d}: "
          f"{CLASSES[pred['actual']]:6s} "
          f"(확신도: {pred['confidence']:.1%})")

# ============================================================================
# 4. Android 비교 가이드
# ============================================================================
print("\n" + "="*80)
print("📱 4단계: Android 앱 테스트 방법")
print("="*80)

print(f"""
다음 단계를 따라하세요:

1️⃣ 파일 복사:
   {OUTPUT_FILENAME} → app/src/main/assets/

2️⃣ MainActivity.kt 수정:
   val totalTestCount = {TOTAL_TEST_COUNT}
   val assetPath = "{OUTPUT_FILENAME}"

3️⃣ Android Studio에서:
   - Clean Project
   - Rebuild Project
   - 앱 실행

4️⃣ "테스트 시작" 버튼 클릭

5️⃣ 결과 비교:

┌─────────────────────────────────────────────────────────┐
│ [Python 기준값]                                          │
│   정확도:      {accuracy * 100:5.2f}%                                      │
│   평균 시간:   {avg_time_ms:5.2f}ms                                     │
│                                                         │
│ [Android 결과] (앱에서 확인)                              │
│   CPU 정확도:  ___.__%                                  │
│   CPU 시간:    ___.___ms                                │
│   GPU 정확도:  ___.__%                                  │
│   GPU 시간:    ___.___ms                                │
│   속도 향상:   ___.___배                                 │
└─────────────────────────────────────────────────────────┘
""")

# ============================================================================
# 5. 결과 저장 (선택)
# ============================================================================
print("\n" + "="*80)
print("💾 5단계: 결과 저장")
print("="*80)

result_filename = f"python_result_{TOTAL_TEST_COUNT}.txt"
with open(result_filename, 'w', encoding='utf-8') as f:
    f.write(f"CIFAR-10 TFLite Python 테스트 결과\n")
    f.write(f"=" * 80 + "\n\n")
    f.write(f"테스트 개수: {TOTAL_TEST_COUNT}\n")
    f.write(f"모델: {MODEL_PATH}\n")
    f.write(f"데이터: {OUTPUT_FILENAME}\n\n")
    f.write(f"정확도: {accuracy * 100:.2f}% ({correct}/{TOTAL_TEST_COUNT})\n")
    f.write(f"평균 추론시간: {avg_time_ms:.2f}ms\n")
    f.write(f"표준편차: {std_time_ms:.2f}ms\n")
    
    if TOTAL_TEST_COUNT >= 100:
        f.write(f"\n95% 신뢰구간: {ci_lower:.2f}% ~ {ci_upper:.2f}%\n")
        f.write(f"오차범위: ±{margin * 100:.2f}%p\n")
    
    f.write(f"\n클래스별 정확도:\n")
    for i in range(10):
        if class_total[i] > 0:
            class_acc = class_correct[i] / class_total[i] * 100
            f.write(f"  {CLASSES[i]:6s}: {class_acc:5.1f}% ({class_correct[i]}/{class_total[i]})\n")

print(f"\n✅ 결과 저장 완료: {result_filename}")

# ============================================================================
# 완료
# ============================================================================
print("\n" + "="*80)
print("✨ 모든 준비 완료!")
print("="*80)

print(f"""
생성된 파일:
  📦 {OUTPUT_FILENAME} ({file_size_mb:.2f} MB) - Android에 복사하세요
  📄 {result_filename} - 비교용 기준값

다음 단계:
  1. {OUTPUT_FILENAME}을 Android 프로젝트의 assets 폴더로 복사
  2. Android 앱 실행
  3. Python 결과와 비교

Happy Testing! 🚀
""")

print("="*80)

🚀 CIFAR-10 온디바이스 AI 테스트 준비 (1000개)

📦 1단계: 테스트 데이터 바이너리 생성

데이터 정보:
  Shape: (1000, 32, 32, 3)
  레이블 범위: 0 ~ 9
  픽셀 범위: 0.000 ~ 1.000

바이너리 파일 생성 중: cifar10_test_1000.bin
  진행: 1/1000 (0.1%)
  진행: 100/1000 (10.0%)
  진행: 200/1000 (20.0%)
  진행: 300/1000 (30.0%)
  진행: 400/1000 (40.0%)
  진행: 500/1000 (50.0%)
  진행: 600/1000 (60.0%)
  진행: 700/1000 (70.0%)
  진행: 800/1000 (80.0%)
  진행: 900/1000 (90.0%)
  진행: 1000/1000 (100.0%)

✅ 생성 완료!
  파일: cifar10_test_1000.bin
  크기: 11.72 MB

🔍 데이터 검증 (처음 3개):
  이미지 #0: 레이블=3 (고양이), 범위=[0.051, 1.000] ✅
  이미지 #1: 레이블=8 (배), 범위=[0.000, 0.969] ✅
  이미지 #2: 레이블=8 (배), 범위=[0.012, 0.988] ✅

🧪 2단계: Python에서 모델 테스트

모델 로드: models/base_int8.tflite

모델 정보:
  Input shape: [ 1 32 32  3]
  Input type: <class 'numpy.uint8'>
  Output shape: [ 1 10]
  Output type: <class 'numpy.uint8'>

추론 시작 (1000개)...
  진행: 100/1000 - 정확도: 85.00% - 평균 시간: 0.49ms
  진행: 200/1000 - 정확도: 83.50% - 평균 시간: 0.47ms
  진행: 300/1000 - 정확도: 80.67% - 평균 시간: 0.47ms
  진행: 400/1000 - 정확도: 80.25% - 평균 시간: